### 189. Rotate Array
Given an integer array `nums`, rotate the array to the right by `k` steps, where `k` is non-negative.

**Example 1:**
```shell
Input: nums = [1,2,3,4,5,6,7], k = 3
Output: [5,6,7,1,2,3,4]
```
**Explanation:**
```shell
rotate 1 steps to the right: [7,1,2,3,4,5,6]
rotate 2 steps to the right: [6,7,1,2,3,4,5]
rotate 3 steps to the right: [5,6,7,1,2,3,4]
```
**Example 2:**
```shell
Input: nums = [-1,-100,3,99], k = 2
Output: [3,99,-1,-100]
```
**Explanation:**
```shell
rotate 1 steps to the right: [99,-1,-100,3]
rotate 2 steps to the right: [3,99,-1,-100]
```
**Constraints:**

- `1 <= nums.length <= 105`
- `-231 <= nums[i] <= 231 - 1`
- `0 <= k <= 105`

**Follow up:**

- Try to come up with as many solutions as you can. There are at least **three** different ways to solve this problem.
- Could you do it in-place with $O(1)$ extra space?

### Reasoning
#### Step 1 - Gather requirements
I'm asked to rotate the given array to the right with `k` steps. The steps defined by `k`, which is **non-negative** integer number.

- Is the input sorted? What order if yes?
- Can the input be empty or null?
- Are there duplicates, negatives and zeros?
- Should the output be ordered?
- What are the bounds of input size?

**1. Is the input sorted?** No. The order is arbitrary.

**2. Can the input be empty or null?** No. `nums` always has at least 1 element.

**3. Duplicates, negatives, zeros?** Yes, all three can appear. Every value fits in a 32-bit signed int, from -2³¹ to 2³¹-1.

**4. Should the output be ordered?** For this problem, the order is the answer. **The result must be exactly the original array shifted right by `k`**. A sharper version of this question is: _"Do I return a new array, or modify the input?"_ Here's that answer:

- You must modify `nums` **in place**.
- The function returns nothing (`None`).
- Follow-up an interviewer will likely ask: can you do it with **$O(1)$ extra space**?

**5. Bounds on input size?**

- `1 <= len(nums) <= 10^5`
- `0 <= k <= 10^5`

Compare the two bounds carefully. They're independent, and that matters.

### Step 2 - Exampes + Edge Cases

```shell
nums = [1, 2, 3, 4, 5, 6, 7], k = 3
Output: [5, 6, 7, 1, 2, 3, 4]
1. [7, 1, 2, 3, 4, 5, 6]
2. [6, 7, 1, 2, 3, 4, 5]
3. [5, 6, 7, 1, 2, 3, 4]
```
```shell
nums = [-1, -100, 3, 99], k = 2
Output: [3, 99, -1, -100]
1. [99, -1, -100, 3]
2. [3, 99, -1, -100]
```

```shell
nums = [1, 2, 3], k = 4
Output: [3, 1, 2]
1. [3, 1, 2]
2. [2, 3, 1]
3. [1, 2, 3]  when k = n the output array is the same
4. [3, 1, 2]  k % n now gives number of rotates that change array
```
```shell
nums = [1, 2, 3, 4], k = 0
Output: [1, 2, 3, 4]
```

```shell
nums = [3], k = 3
Output: [3]
```

> **Property 1**: `k = n` brings the array back to where it started is the **key insight** here.

> **Property 2**: every full lap of `n` rotations **cancels out**, and only the leftover rotations matter.

`k` gives the number of rotation to be done. We know if `n = k` makes a full cycle and ends up at the initial state. Once `k` starts to grow over `n`, we can say that another cycle starts **from scratch**. So, we can use modulo `%` operation, and see how much `n` goes into `k`, this will cancel full cycles out and leave only the number of rotations to be done. It is `k % n`. In this example `4 % 3 = 1`, one rotate to the right.

> `k % n` removes the **full cycles** and leaves only the rotations that actually change the array. It also handles `k = 0` safely, because the divisor is now `n`, which is always at least 1.

### Step 3 - Brute Force - `k % n` + `pop` + `insert`

`k % n` **cancels** out full cycles of rotation, leaving the true number of rotations that actually **change** the input array.

Using `k % n` instead of "n times" matters. When `k` is a multiple of `n`, this code does zero work.

![Visual](../../images/189rotate_array_brute_force.png)

In [ ]:
def brute(nums: list[int], k: int) -> None:
    # brute force - k % n + .pop() + .insert()
    # setup
    # set the number of rotation times: rot_times = k % len(nums)
    # scan input - for loop - [0, k % n]

    # 1. pop and save the last item: last = nums.pop()
    # 2. insert the last item at the head of the array

    k = k % len(nums)

    for _ in range(k):  # O(k % n)
        last = nums.pop()  # O(1)
        nums.insert(0, last)  # O(n)  shift all items, n ops

In [6]:
data = [1, 2, 3, 4, 5, 6, 7]
brute(data, 3)
data  # [5, 6, 7, 1, 2, 3, 4]

[5, 6, 7, 1, 2, 3, 4]

In [5]:
data = [-1, -100, 3, 99]
brute(data, 2)
data  # [3, 99, -1, -100]

[3, 99, -1, -100]

In [7]:
data = [1, 2, 3]
brute(data, 4)
data # [3, 1, 2]

[3, 1, 2]

In [8]:
data = [1, 2, 3, 4]
brute(data, 0)
data # [1, 2, 3, 4]

[1, 2, 3, 4]

In [9]:
data = [3]
brute(data, 3)
data # [3]

[3]

`pop()` is $O(1)$ and `insert(0, ...)` is $O(n)$, so each rotation costs $O(n)$.

- **Time**: $O(n\space \times (k\space \text mod\space n))$, which is $O(n²)$ in the worst case. The loop runs `k % n` times, at most `n-1`, and each `insert(0)` shifts all `n` elements.
- **Space**: $O(1)$. Only one saved element; the rotation happens in place.

**Find the redundant work**

In [11]:
brute([1, 2, 3, 4, 5, 6, 7], 6)  # [5, 6, 7, 1, 2, 3, 4]

Element `1` gets shifted 3 times, one per rotation.
If I already knew where it belonged, I could write it straight to index 3.

> **Redundant Work**: The Brute force moves every element `k` times, one step per rotation, when one direct move per element would do.

> **Property**: elements `1` starts at index 0, gets rotated 3 times, and lands at index 3. Let's turn it into a rule.

In the same example, element `5` starts at index 4 and ends at index 0:
```shell
index:   0  1  2  3  4  5  6
before: [1, 2, 3, 4, 5, 6, 7]
after:  [5, 6, 7, 1, 2, 3, 4]
```
![Visual](../../images/189rotate_array_formula.png)

### Step 4 - Select Algorithm

Allocate a second array, put each element at `(i + k) % n` then copy back into `nums`.

> Each element moves **once**, straight to `(i + k) % n`.

- **Time**: $O(n\times 2)$ (drop constants - $O(n)$) with 2 passes, first writes the second array, then copy back into `nums` from the resulting array.
- **Space**: $O(n)$ since the extra array of `n` items is stored for the second array. Copying back into `nums` doesn't use any extra space; it just reuses memory it already has.

**Space is traded** for time. The $O(1)$ space follow-up is still needed next.

![Visual](../../images/189rotate_array_extra_space.png)

### Step 5 - Function Siganture + Pseudocode
**2 Passes over `n` elements** - which is $O(2n)$ operations.
```py
def rotate_arr(nums: list[int], k: int) -> None:
    # 2 passes (with 2n ops) - (i + k) % n for indices in the extra array
    # setup
    # allocate an extra array of n length with None placeholders: res = [None] * len(nums)
    # scan input array - for loop - full pass

    # 1. compute the new index with (i + k) % n to copy elements into res
    # 2. copy items from nums to res using the new index

    # scan res array - for loop - full pass

    # 3. copy elements from res to nums back
```

### Step 7 - Write Code

In [13]:
def rotate_arr(nums: list[int], k: int) -> None:
    n = len(nums)
    res = [None] * n

    for i in range(n):
        res_idx = (i + k) % n

        res[res_idx] = nums[i]

    # for i in range(len(res)):
    #     nums[i] = res[i]

    nums[:] = res

### Step 8 - Test Examples + Edge cases

In [26]:
data1 = [1, 2, 3, 4, 5, 6, 7]

rotate_arr(data1, k=3)  # [5, 6, 7, 1, 2, 3, 4]
print(data1)

[5, 6, 7, 1, 2, 3, 4]


In [9]:
data2 = [-1, -100, 3, 99]

rotate_arr(data2, k=2)  # [3, 99, -1, -100]
print(data2)

[3, 99, -1, -100]


In [10]:
data3 = [1, 2, 3]

rotate_arr(data3, k=4)  # [3, 1, 2]
print(data3)

[3, 1, 2]


In [11]:
data4 = [1, 2, 3, 4]

rotate_arr(data4, k=0)  # [1, 2, 3, 4]
print(data4)

[1, 2, 3, 4]


In [12]:
data5 = [3]

rotate_arr(data5, k=3)
print(data5)

[3]


Using `None` as the placeholder is a reasonable choice: if a slot never got filled, you'd see `None` in the output rather than a misleading `0`.

**Note** inside the function:
```shell
nums[i] = x      → mutates the object   (caller sees it)
nums = other     → rebinds the name     (caller doesn't)
```

> **Slice Assignment**: Python also has a **one-line way to mutate in place**: `nums[:] = res`. It's slice assignment, so it writes into the existing list instead of rebinding the name. It does the same thing as your copy-back loop.

### Step 9 - State Final Time & Space complexity
- **Time**: $O(n\times 2)$ (drop constants - $O(n)$) with 2 passes, first writes the second array, then copy back into `nums` from the resulting array.
- **Space**: $O(n)$ since the extra array of `n` items is stored for the second array. Copying back into `nums` doesn't use any extra space; it just reuses memory it already has.

#### Reverse Algorithm

***Follow-up**: "Can you do it in $O(1)$ extra space?"

Look at the before and after as two blocks:
```shell
before: [1, 2, 3, 4 | 5, 6, 7]
after:  [5, 6, 7 | 1, 2, 3, 4]
```
Reverse the entire `before` array by hand and compare it to `after`. What do you notice about the two blocks?
```shell
reversed: [7, 6, 5 | 4, 3, 2, 1]
after:    [5, 6, 7 | 1, 2, 3, 4]
```
> **Property**: Reversing the whole array **puts each block in the correct position**: `[7, 6, 5]` is now **at the front**, where `[5, 6, 7]` belongs.

The order inside the block is **still backwards**, though.

Look at the second block the same way:
```shell
reversed: [7, 6, 5 | 4, 3, 2, 1]
after:    [5, 6, 7 | 1, 2, 3, 4]
```

1. What single operation could you apply to each block separately to turn `reversed` into `after`?
2. And which index boundaries define the two blocks, in terms of `k` and `n`?

Correct, that is the **Reversal Algorithm**
```shell
reversed[:k] -> reverse
reversed[k:] -> reverse
```

```shell
nums = [1, 2, 3], k = 4
k = k % n

1. reverse - [3, 2, 1]
2. revesre - nums[:k] from 0 to 1 - [3]
3. reverse - nums[k:] from 1 to end -> [1, 2]
```

#### Reverse Helper - Makes Revese In-Place

Swap two elements from **both ends** of the array. Two pointers converging.

![Visual](../../images/189rotate_array_rev_helper.png)

In [ ]:
def reverse_items(arr: list[int], left: int, right: int) -> None:
    # helper - reverse array
    # setup
    # set two pointers at the ends of the array
    # scan input: while loop - full pass - two pointers at the ends

    # 1. swap left and right values
    # 2. move pointers converging
    while left < right:
        arr[left], arr[right] = arr[right], arr[left]
        left += 1
        right -= 1


def rotate_arr2(nums: list[int], k: int) -> None:
    # time O(n), space O(1) - helper to reverse the array's subarrays
    # setup
    # define k that rotates & changes array: k = k % n

    # 1. reverse the entire input array
    # 2. reverse [0:k] elements
    # 3. reverse [k:] elements

    n = len(nums)
    k = k % n

    reverse_items(nums, left=0, right=n - 1)
    reverse_items(nums, left=0, right=k - 1)
    reverse_items(nums, left=k, right=n - 1)

In [16]:
data1 = [1, 2, 3, 4, 5, 6, 7]
rotate_arr2(data1, 3)
print(data1)

[5, 6, 7, 1, 2, 3, 4]


In [17]:
data2 = [-1, -100, 3, 99]

rotate_arr2(data2, k=2)  # [3, 99, -1, -100]
print(data2)

[3, 99, -1, -100]


In [18]:
data3 = [1, 2, 3]

rotate_arr2(data3, k=4)  # [3, 1, 2]
print(data3)

[3, 1, 2]


In [20]:
data4 = [1, 2, 3, 4]

rotate_arr2(data4, k=0)  # [1, 2, 3, 4]
print(data4)

[1, 2, 3, 4]


In [21]:
data5 = [3]

rotate_arr2(data5, k=3)
print(data5)

[3]


- **Time**: $O(n)$ with 3 passes, one is `n/2`, `k/2` + `(n - k)/2` for swaps.
```shell
whole array:   n/2 swaps
first block:   k/2 swaps
second block:  (n - k)/2 swaps
total:         n/2 + n/2 = n swaps → O(n)
```
- **Space**: No extra space is used apart from a couple of scalars.

#### Final Comparison
```shell
Brute force (pop/insert):  O(n · (k mod n)) time, O(1) space
Extra array:               O(n) time,           O(n) space
Triple reversal:           O(n) time,           O(1) space  ← optimal on both
```

### Session wrap-up

- `k % n` reduces any rotation count to the rotations that actually change the array. Choose the divisor by asking "what's the cycle length?"
- `(i + k) % n` is a single formula for an element's wrapped index, with no special cases.
- Rebinding a name (`nums = res`) is different from mutating an object (`nums[i] = x`), and slicing creates a copy.
- Triple reversal rotates an array in place.

**Patterns:** **Two pointers** (converging, for in-place reversal) and **modular index arithmetic** (circular arrays). The reversal trick is worth filing on its own: _reverse the whole, then reverse the parts_ rearranges blocks in place.